## 1. Dataset Overview

The dataset has a total of 38113 rows and 81 columns with 38113 rows having at least 1 NA

In [173]:
import pandas as pd

df_raw = pd.read_csv('../data/fuel.csv')

print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])

print('\nColumn names:')
for i in df_raw.columns.tolist():
    print(i)

print('\nMissing values (only columns that have some):')
missing = df_raw.isnull().sum()
missing = missing[missing > 0]
print(missing)
print("Total rows with at least one NA:", df_raw.isnull().any(axis=1).sum())

print("\nUnique value counts: \n", df_raw.nunique())

print('\ndtypes:')
print(df_raw.dtypes)

Rows: 38113
Columns: 81

Column names:
vehicle_id
year
make
model
class
drive
transmission
transmission_type
engine_index
engine_descriptor
engine_cylinders
engine_displacement
turbocharger
supercharger
fuel_type
fuel_type_1
fuel_type_2
city_mpg_ft1
unrounded_city_mpg_ft1
city_mpg_ft2
unrounded_city_mpg_ft2
city_gasoline_consumption_cd
city_electricity_consumption
city_utility_factor
highway_mpg_ft1
unrounded_highway_mpg_ft1
highway_mpg_ft2
unrounded_highway_mpg_ft2
highway_gasoline_consumption_cd
highway_electricity_consumption
highway_utility_factor
unadjusted_city_mpg_ft1
unadjusted_highway_mpg_ft1
unadjusted_city_mpg_ft2
unadjusted_highway_mpg_ft2
combined_mpg_ft1
unrounded_combined_mpg_ft1
combined_mpg_ft2
unrounded_combined_mpg_ft2
combined_electricity_consumption
combined_gasoline_consumption_cd
combined_utility_factor
annual_fuel_cost_ft1
annual_fuel_cost_ft2
gas_guzzler_tax
save_or_spend_5_year
annual_consumption_in_barrels_ft1
annual_consumption_in_barrels_ft2
tailpipe_co2_ft

C:\Users\theor\AppData\Local\Temp\ipykernel_41076\3761585670.py:3: DtypeWarning: Columns (7: transmission_type, 44: gas_guzzler_tax) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('../data/fuel.csv')


## 2. Data Cleanup and Creation

We selected a smaller set of attributes relevant to the prediction task. Other attributes were excluded because they were not relevant to the first model, contained excessive missing values, or could introduce target leakage by directly revealing fuel consumption.

In [160]:
master_cols = [
    'year',
    'make',
    'model',
    'class',
    'drive',
    'transmission',
    'transmission_type',
    'engine_cylinders',
    'engine_displacement',
    'turbocharger',
    'supercharger',
    'fuel_type',
    'fuel_type_1',
    'engine_descriptor',
    'start_stop_technology',
    'alternative_fuel_technology',
    'electric_motor',
    'combined_mpg_ft1'
]

df = df_raw[master_cols].copy()

### Target Column Creation

We can use the combined_mpg_ft1 feature and the L/100km = 235.215 / MPG formula to create the target column which we will evaluate our model against. Before calculating the target, we check that `combined_mpg_ft1` is greater than zero, since zero or negative MPG values would be invalid and would make the conversion inappropriate.

In [161]:
df = df[df["combined_mpg_ft1"] > 0].copy()

df["combined_l_per_100km"] = 235.215 / df["combined_mpg_ft1"]

### Post Investigation - What Rows are Dropped?

At this stage, electric, hybrid and alternative-fuel vehicles are temporarily retained. They are filtered later once the missing-value and feature-selection steps are complete.

The `electric_motor`, `start_stop_technology` and `supercharger` features contain only NA values, so we remove these columns. We also remove `transmission_type` as it has many missing values and is not essential for our first model.

`fuel_type_1` appears to be a duplicate of `fuel_type`, and `alternative_fuel_technology` is mostly irrelevent once we filter to conventional fuels.



In [162]:
df = df.drop(columns=[
    'electric_motor',
    'start_stop_technology',
    'supercharger',
    'transmission_type',
    'fuel_type_1',
    'alternative_fuel_technology'
])

 ### Feature Investigation: Turbocharger
 
The field contains either `True` or `missing values (NA)`, so for this initial dataset we treat missing values as `False`, assuming a blank indicates no turbocharger.

In [163]:
df['turbocharger'] = df['turbocharger'].notna()
print(df.isnull().sum())

year                        0
make                        0
model                       0
class                       0
drive                    1189
transmission               11
engine_cylinders          136
engine_displacement       134
turbocharger                0
fuel_type                   0
engine_descriptor       15420
combined_mpg_ft1            0
combined_l_per_100km        0
dtype: int64


### Missing Values Investigation
There are 1319 rows missing at least one of the essential features (`drive`, `transmission`, `engine_cylinders` or `engine_displacement`). We remove these rows, keeping the optional columns even where some values are missing.

In [164]:
essential_cols = [
    'drive',
    'transmission',
    'engine_cylinders',
    'engine_displacement'
]

df = df.dropna(subset=essential_cols).copy()
print('Rows remaining:', len(df))
print('Missing values remaining:')
print(df.isna().sum()[lambda s: s > 0])

Rows remaining: 36794
Missing values remaining:
engine_descriptor    15263
dtype: int64


### Fuel Type Investigation
For the first version of the model, we restrict the dataset to conventional petrol and diesel vehicles. Electric, hybrid and other alternative-fuel vehicles use different measures of energy consumption and would make the target less directly comparable. We therefore retain Regular, Premium, Midgrade and Diesel fuel types.

In [165]:
conventional_fuels = [
    'Regular',
    'Premium',
    'Midgrade',
    'Diesel'
]

df_model = df[df['fuel_type'].isin(conventional_fuels)].copy()


### Duplicate Value Investigation
Exact duplicate rows are identified and removed.

In [166]:
# Check exact duplicate rows
print("Exact duplicate rows:", df_model.duplicated().sum())

# Remove exact duplicate rows
before = len(df_model)

df_model = df_model.drop_duplicates().copy()

after = len(df_model)

print("Rows removed:", before - after)
print("Rows remaining:", after)

Exact duplicate rows: 318
Rows removed: 318
Rows remaining: 34978



### Duplicate Value Investigation (Continued)
Check for repeated vehicle configurations.

Many apparently similar vehicles differed in other attributes, particularly engine_descriptor, and sometimes had different fuel-consumption values. These records were therefore retained because they may represent genuine vehicle variants rather than duplicate data.

In [167]:
config_cols = [
    "year",
    "make",
    "model",
    "engine_cylinders",
    "engine_displacement",
    "transmission",
    "drive",
    "fuel_type",
]

repeated_configs = df_model[
    df_model.duplicated(subset=config_cols, keep=False)
]

config_target_summary = (
    repeated_configs
    .groupby(config_cols)["combined_l_per_100km"]
    .agg(["count", "nunique"])
    .reset_index()
)

different_targets = config_target_summary[
    config_target_summary["nunique"] > 1
]

print("Rows in repeated configurations:", len(repeated_configs))
print("Unique repeated configurations:", repeated_configs.groupby(config_cols).ngroups)
print("Configurations with multiple target values:",len(different_targets))

Rows in repeated configurations: 4499
Unique repeated configurations: 2130
Configurations with multiple target values: 1438


### Check Numeric Values and Potential Outliers

The main numerical attributes were checked for invalid and extreme values. No zero or negative values were found for MPG, engine displacement or engine cylinder count. The minimum and maximum values were unusual in some cases, but remained plausible for real vehicles, so no numerical outliers were removed at this stage.

In [168]:
numeric_cols = [
    "year",
    "engine_cylinders",
    "engine_displacement",
    "combined_mpg_ft1",
    "combined_l_per_100km",
]

print(df_model[numeric_cols].describe())
print("MPG <= 0:", (df_model["combined_mpg_ft1"] <= 0).sum())
print("Engine displacement <= 0:", (df_model["engine_displacement"] <= 0).sum())
print("Engine cylinders <= 0:", (df_model["engine_cylinders"] <= 0).sum())

               year  engine_cylinders  engine_displacement  combined_mpg_ft1  \
count  34978.000000      34978.000000         34978.000000      34978.000000   
mean    2000.325233          5.713277             3.294508         19.980159   
std       10.168858          1.745275             1.346244          5.055085   
min     1984.000000          2.000000             0.900000          7.000000   
25%     1991.000000          4.000000             2.200000         17.000000   
50%     2001.000000          6.000000             3.000000         19.000000   
75%     2009.000000          6.000000             4.300000         23.000000   
max     2017.000000         16.000000             8.400000         58.000000   

       combined_l_per_100km  
count          34978.000000  
mean              12.507259  
std                3.125923  
min                4.055431  
25%               10.226739  
50%               12.379737  
75%               13.836176  
max               33.602143  
MPG <= 0:

### Check Categorical Consistency

This cell checks for inconsistencies across the file, most notably naming. This dataset ranges over many years, meaning naming conventions could change over time, so it's important to check this so the model doesn't assume different relationships for the same category.

In [169]:
categorical_cols = [
    "class",
    "drive",
    "transmission",
    "fuel_type",
]

for col in categorical_cols:
    print(f"\n{col}")
    print(df_model[col].value_counts(dropna=False))

class_years = (
    df_model
    .groupby("class")["year"]
    .agg(["count", "min", "max"])
    .sort_values("min")
)

print(class_years.to_string())

print("DRIVE")
print(df_model["drive"].value_counts().to_string())

print("\nTRANSMISSION")
transmission_summary = (
    df_model
    .groupby("transmission")["year"]
    .agg(["count", "min", "max"])
    .sort_index()
)

print(transmission_summary.to_string())

pickup_classes = df_model[
    df_model["class"].str.contains("Pickup", case=False, na=False)
]

print(
    pickup_classes.groupby("class")["year"]
    .agg(["count", "min", "max"])
    .to_string()
)

suv_classes = df_model[
    df_model["class"].str.contains("Utility", case=False, na=False)
]

print(
    suv_classes.groupby("class")["year"]
    .agg(["count", "min", "max"])
    .to_string()
)


class
class
Compact Cars                          5112
Subcompact Cars                       4391
Midsize Cars                          4005
Standard Pickup Trucks                2266
Sport Utility Vehicle - 4WD           1943
Two Seaters                           1802
Large Cars                            1680
Sport Utility Vehicle - 2WD           1457
Special Purpose Vehicles              1428
Small Station Wagons                  1372
Minicompact Cars                      1223
Vans                                  1134
Standard Pickup Trucks 2WD             995
Standard Pickup Trucks 4WD             824
Midsize-Large Station Wagons           625
Special Purpose Vehicle 2WD            586
Small Pickup Trucks                    537
Small Sport Utility Vehicle 4WD        502
Small Pickup Trucks 2WD                419
Midsize Station Wagons                 413
Small Sport Utility Vehicle 2WD        367
Standard Sport Utility Vehicle 4WD     347
Vans, Cargo Type                       32

### Standardise Categorical Inconsistencies

Inspection of the categorical attributes identified a small number of inconsistent representations. Obvious formatting variations were standardised, such as "Manual 5 Speed" and "Manual 5-Speed", and several minor vehicle-class naming variations. Similar-looking categories were not automatically merged where they could represent genuine differences or changes in the source classification system.

In [170]:
class_replacements = {
    "Standard Pickup Trucks/2wd": "Standard Pickup Trucks 2WD",
    "Special Purpose Vehicles/2wd": "Special Purpose Vehicle 2WD",
    "Special Purpose Vehicles/4wd": "Special Purpose Vehicle 4WD",
}

df_model["class"] = df_model["class"].replace(class_replacements)

transmission_replacements = {
    "Manual 5 Speed": "Manual 5-Speed",
    "Manual(M7)": "Manual 7-Speed",
    "Automatic 6Speed": "Automatic 6-Speed",
    "Auto (AV-S6)": "Auto(AV-S6)",
    "Auto (AV-S8)": "Auto(AV-S8)",
}

df_model["transmission"] = df_model["transmission"].replace(
    transmission_replacements
)

df_model["class"] = df_model["class"].replace({
    "Vans Passenger": "Vans, Passenger Type"
})

print("Class categories:", df_model["class"].nunique())
print("Transmission categories:", df_model["transmission"].nunique())

print("\nMissing values remaining:")
print(df_model.isna().sum()[lambda s: s > 0])


Class categories: 30
Transmission categories: 38

Missing values remaining:
engine_descriptor    15067
dtype: int64


## 3. Final CSV File

After completing the cleaning checks and transformations, the resulting dataset is saved as `cleaned_fuel.csv`. This file will be used as the cleaned source for the modelling stage.

In [171]:
from pathlib import Path

output_dir = Path('../preprocessed_data')
output_dir.mkdir(parents=True, exist_ok=True)

df_model.to_csv(output_dir / 'cleaned_fuel.csv', index=False)

print(f"Saved {len(df_model)} rows and {len(df_model.columns)} columns")
print("Exact duplicates remaining:", df_model.duplicated().sum())

Saved 34978 rows and 13 columns
Exact duplicates remaining: 0
